# 🔄 Notebook 1 — Ingesta y Limpieza de Datos
## Caso: Predicción de Default en Solicitudes de Préstamos

### Objetivo
Realizar la ingesta del dataset de préstamos, limpiarlo y transformarlo
para dejarlo listo para la etapa de validación.

### Etapas de este notebook
1. Ingesta del CSV a PostgreSQL
2. Lectura desde PostgreSQL con pandas
3. Limpieza de datos
4. Transformación
5. Exportación del dataset limpio

In [1]:
import pandas as pd
import os
import logging
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

print("Librerías cargadas correctamente")

Librerías cargadas correctamente


In [3]:
# Crear la carpeta logs si no existe (exist_ok evita error si ya existe)
os.makedirs("../logs", exist_ok=True)

# Configurar el sistema de logs
logging.basicConfig(
    filename="../logs/ingest.log",  # archivo donde se guardan los logs
    level=logging.INFO,             # nivel de detalle: INFO registra eventos normales
    format="%(asctime)s - %(levelname)s - %(message)s"  # formato: fecha - tipo - mensaje
)

# Registrar el inicio del proceso en el log
logging.info("Inicio del proceso de ingesta")

print("Logger configurado correctamente")

Logger configurado correctamente


In [4]:
# Cargar las variables de entorno desde el archivo .env
load_dotenv("../.env")

# Leer las credenciales de la base de datos desde el .env
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

# Crear la cadena de conexión a PostgreSQL
connection_string = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

# Crear el engine de conexión
engine = create_engine(connection_string)

# Verificar que la conexión funciona
try:
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))
        logging.info("Conexión a PostgreSQL exitosa")
        print("Conexión a PostgreSQL exitosa")
except Exception as e:
    logging.error(f"Error al conectar: {e}")
    print(f"Error al conectar: {e}")

Conexión a PostgreSQL exitosa


In [6]:
# Leer el CSV original desde la carpeta data/raw
df_raw = pd.read_csv("/workspaces/dataops-loan-pipeline/data/raw/02_loan_data.csv")

# Registrar en el log cuántos registros se leyeron
logging.info(f"CSV leído correctamente: {len(df_raw)} registros")
print(f"CSV leído: {len(df_raw)} registros")

# Cargar el CSV a PostgreSQL como tabla loans_raw (si ya existe la reemplaza)
df_raw.to_sql("loans_raw", engine, if_exists="replace", index=False)

# Registrar en el log que la carga fue exitosa
logging.info("Tabla loans_raw creada en PostgreSQL")
print("Tabla loans_raw cargada en PostgreSQL")

CSV leído: 45000 registros
Tabla loans_raw cargada en PostgreSQL


In [8]:
# Leer los datos desde PostgreSQL para confirmar la ingesta
df = pd.read_sql("SELECT * FROM loans_raw", engine)

# Registrar en el log
logging.info(f"Datos leídos desde PostgreSQL: {len(df)} registros")
print(f"Datos leídos desde PostgreSQL: {len(df)} registros")

# Ver las primeras filas
df.head()

Datos leídos desde PostgreSQL: 45000 registros


,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1


In [9]:
# Exploración inicial del dataset antes de limpiar
print("Información general del dataset:")
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")
print()
print("Tipos de datos:")
print(df.dtypes)
print()
print("Valores nulos por columna:")
print(df.isnull().sum())
print()
print(f"Duplicados: {df.duplicated().sum()}")

Información general del dataset:
Filas: 45000
Columnas: 14

Tipos de datos:
person_age                        float64
person_gender                         str
person_education                      str
person_income                     float64
person_emp_exp                      int64
person_home_ownership                 str
loan_amnt                         float64
loan_intent                           str
loan_int_rate                     float64
loan_percent_income               float64
cb_person_cred_hist_length        float64
credit_score                        int64
previous_loan_defaults_on_file        str
loan_status                         int64
dtype: object

Valores nulos por columna:
person_age                        0
person_gender                     0
person_education                  0
person_income                     0
person_emp_exp                    0
person_home_ownership             0
loan_amnt                         0
loan_intent                       0
loan_i

In [10]:
# Convertir columnas que deberían ser enteras de float a int
df["person_age"] = df["person_age"].astype(int)
df["person_income"] = df["person_income"].astype(int)
df["loan_amnt"] = df["loan_amnt"].astype(int)
df["cb_person_cred_hist_length"] = df["cb_person_cred_hist_length"].astype(int)

# Registrar en el log
logging.info("Tipos de datos corregidos correctamente")
print("Tipos de datos corregidos")
print()
print(df.dtypes)

Tipos de datos corregidos

person_age                          int64
person_gender                         str
person_education                      str
person_income                       int64
person_emp_exp                      int64
person_home_ownership                 str
loan_amnt                           int64
loan_intent                           str
loan_int_rate                     float64
loan_percent_income               float64
cb_person_cred_hist_length          int64
credit_score                        int64
previous_loan_defaults_on_file        str
loan_status                         int64
dtype: object


In [11]:
# Guardar el dataset limpio como CSV para la siguiente etapa
df.to_csv("../data/processed/loans_clean.csv", index=False)

# Registrar en el log
logging.info("Dataset limpio guardado en data/processed/loans_clean.csv")
logging.info("Fin del proceso de ingesta y limpieza")
print("Dataset limpio guardado en data/processed/loans_clean.csv")

Dataset limpio guardado en data/processed/loans_clean.csv
